<a href="https://colab.research.google.com/github/ganthantm65/smart_bin_user_module/blob/main/waste_detection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import os
import shutil

# The path we found from your debug output
root_data_dir = "/content/dataset_raw/Organized_Waste_Dataset"
final_path = "/content/waste_data"

categories = ['BD', 'E_Waste', 'NBD', 'Medical']
os.makedirs(final_path, exist_ok=True)

for cat in categories:
    target_dir = os.path.join(final_path, cat)
    os.makedirs(target_dir, exist_ok=True)
    src_dir = os.path.join(root_data_dir, cat)

    # Flatten everything (including subfolders in Medical) into the target
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Create a unique name if there are duplicates across subfolders
                src_file = os.path.join(root, file)
                # Use the path as part of the name to ensure uniqueness
                unique_name = root.replace("/", "_") + "_" + file
                shutil.copy(src_file, os.path.join(target_dir, unique_name))

print("✅ Success: Dataset is flattened!")
!ls /content/waste_data

✅ Success: Dataset is flattened!
BD  E_Waste  Medical  NBD


In [10]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# 1. Image Augmentation (The secret to robustness)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,      # Handle waste dropped at any angle
    brightness_range=[0.7, 1.3], # Handle low light inside the bin
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    '/content/waste_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_generator = train_datagen.flow_from_directory(
    '/content/waste_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# 2. Model Definition
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 3. Train
model.fit(train_generator, validation_data=val_generator, epochs=15)

# 4. Save for local use
model.save('powernest_expert_v1.keras')

Found 15365 images belonging to 4 classes.
Found 3840 images belonging to 4 classes.
Epoch 1/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 329s 661ms/step - accuracy: 0.9103 - loss: 0.2481 - val_accuracy: 0.5531 - val_loss: 1.1648
Epoch 2/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 306s 636ms/step - accuracy: 0.9402 - loss: 0.1643 - val_accuracy: 0.5289 - val_loss: 1.4793
Epoch 3/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 328s 648ms/step - accuracy: 0.9506 - loss: 0.1381 - val_accuracy: 0.5172 - val_loss: 1.4463
Epoch 4/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 293s 610ms/step - accuracy: 0.9513 - loss: 0.1331 - val_accuracy: 0.5604 - val_loss: 1.2694
Epoch 5/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 298s 619ms/step - accuracy: 0.9559 - loss: 0.1224 - val_accuracy: 0.6456 - val_loss: 1.1812
Epoch 6/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 301s 627ms/step - accuracy: 0.9594 - loss: 0.1130 - val_accuracy: 0.5901 - val_loss: 1.2974
Epoch 7/15
481/481 ━━━━━━━━━━━━━━━━━━━━ 306s 636ms/step - accuracy: 0.9617 - loss: 0.1082 - val_accuracy: 0.6737 - val_loss

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

# 1. Define the Data Generator again (to fix the NameError)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# This creates the val_generator variable in memory
val_generator = train_datagen.flow_from_directory(
    '/content/waste_data',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False  # CRITICAL for Confusion Matrix accuracy
)

# 2. Now run the analysis
val_generator.reset()
predictions = model.predict(val_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes
class_names = list(val_generator.class_indices.keys())

# 3. Plot the Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Waste Classification Analysis')
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.show()

Found 3840 images belonging to 4 classes.


NameError: name 'model' is not defined